In [2]:
from pathlib import Path

import pandas as pd
import numpy as np

In [3]:
PROJECT_ROOT = Path("..")

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT.resolve())
print("Raw data path:", DATA_RAW.resolve())
print("Processed data path:", DATA_PROCESSED.resolve())

Project root: /workspaces/potential-pathway-index
Raw data path: /workspaces/potential-pathway-index/data/raw
Processed data path: /workspaces/potential-pathway-index/data/processed


In [4]:
required_files = {
    "base_student_dataset": DATA_PROCESSED / "base_student_dataset.csv",
    "student_vle": DATA_RAW / "studentVle.csv",
    "student_assessment": DATA_RAW / "studentAssessment.csv",
    "assessments": DATA_RAW / "assessments.csv"
}

for file_label, file_path in required_files.items():
    print(f"{file_label:25} | exists: {file_path.exists()} | path: {file_path}")

base_student_dataset      | exists: True | path: ../data/processed/base_student_dataset.csv
student_vle               | exists: True | path: ../data/raw/studentVle.csv
student_assessment        | exists: True | path: ../data/raw/studentAssessment.csv
assessments               | exists: True | path: ../data/raw/assessments.csv


In [5]:
base_student_dataset = pd.read_csv(required_files["base_student_dataset"])

print("base_student_dataset loaded successfully.")
print("Shape:", base_student_dataset.shape)

base_student_dataset.head()

base_student_dataset loaded successfully.
Shape: (32593, 14)


,id_student,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,date_registration,final_result,at_risk_misalignment
0,11391,AAA,2013J,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,-159.0,Pass,0
1,28400,AAA,2013J,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,-53.0,Pass,0
2,30268,AAA,2013J,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,-92.0,Withdrawn,1
3,31604,AAA,2013J,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,-52.0,Pass,0
4,32885,AAA,2013J,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,-176.0,Pass,0


In [6]:
student_vle_columns = [
    "code_module",
    "code_presentation",
    "id_student",
    "date",
    "sum_click"
]

student_vle = pd.read_csv(
    required_files["student_vle"],
    usecols=student_vle_columns
)

print("student_vle loaded successfully.")
print("Shape:", student_vle.shape)

student_vle.head()

student_vle loaded successfully.
Shape: (10655280, 5)


,code_module,code_presentation,id_student,date,sum_click
0,AAA,2013J,28400,-10,4
1,AAA,2013J,28400,-10,1
2,AAA,2013J,28400,-10,1
3,AAA,2013J,28400,-10,11
4,AAA,2013J,28400,-10,1


In [7]:
student_assessment = pd.read_csv(required_files["student_assessment"])

assessments = pd.read_csv(required_files["assessments"])

print("student_assessment loaded successfully.")
print("Shape:", student_assessment.shape)

print("\nassessments loaded successfully.")
print("Shape:", assessments.shape)

student_assessment loaded successfully.
Shape: (173912, 5)

assessments loaded successfully.
Shape: (206, 6)


In [8]:
main_keys = ["id_student", "code_module", "code_presentation"]

EARLY_WINDOW_START = 0
EARLY_WINDOW_END = 30

print("Main keys:", main_keys)
print("Early observation window:", EARLY_WINDOW_START, "to", EARLY_WINDOW_END)

Main keys: ['id_student', 'code_module', 'code_presentation']
Early observation window: 0 to 30


In [9]:
base_total_rows = base_student_dataset.shape[0]
base_unique_keys = base_student_dataset[main_keys].drop_duplicates().shape[0]

print("base_student_dataset total rows:", base_total_rows)
print("base_student_dataset unique student-module-presentation keys:", base_unique_keys)
print("Is the main key unique in base_student_dataset?", base_total_rows == base_unique_keys)

base_student_dataset total rows: 32593
base_student_dataset unique student-module-presentation keys: 32593
Is the main key unique in base_student_dataset? True


In [10]:
print("student_vle date minimum:", student_vle["date"].min())
print("student_vle date maximum:", student_vle["date"].max())

print("\nNumber of rows before course start:")
print((student_vle["date"] < 0).sum())

print("\nNumber of rows inside early window:")
print(student_vle["date"].between(EARLY_WINDOW_START, EARLY_WINDOW_END).sum())

print("\nNumber of rows after early window:")
print((student_vle["date"] > EARLY_WINDOW_END).sum())

student_vle date minimum: -25
student_vle date maximum: 269

Number of rows before course start:
688988

Number of rows inside early window:
2291587

Number of rows after early window:
7674705


In [11]:
print("student_assessment date_submitted minimum:", student_assessment["date_submitted"].min())
print("student_assessment date_submitted maximum:", student_assessment["date_submitted"].max())

print("\nNumber of submissions before course start:")
print((student_assessment["date_submitted"] < 0).sum())

print("\nNumber of submissions inside early window:")
print(student_assessment["date_submitted"].between(EARLY_WINDOW_START, EARLY_WINDOW_END).sum())

print("\nNumber of submissions after early window:")
print((student_assessment["date_submitted"] > EARLY_WINDOW_END).sum())

student_assessment date_submitted minimum: -11
student_assessment date_submitted maximum: 608

Number of submissions before course start:
2057

Number of submissions inside early window:
24465

Number of submissions after early window:
147390


In [12]:
print("assessments due date minimum:", assessments["date"].min())
print("assessments due date maximum:", assessments["date"].max())

print("\nMissing assessment due dates:")
print(assessments["date"].isnull().sum())

print("\nAssessment types:")
print(assessments["assessment_type"].value_counts())

assessments due date minimum: 12.0
assessments due date maximum: 261.0

Missing assessment due dates:
11

Assessment types:
assessment_type
TMA     106
CMA      76
Exam     24
Name: count, dtype: int64


In [13]:
early_student_vle = student_vle[
    student_vle["date"].between(EARLY_WINDOW_START, EARLY_WINDOW_END)
].copy()

print("early_student_vle created successfully.")
print("Shape:", early_student_vle.shape)

early_student_vle.head()

early_student_vle created successfully.
Shape: (2291587, 5)


,code_module,code_presentation,id_student,date,sum_click
10847,AAA,2013J,345357,0,3
10848,AAA,2013J,345357,0,1
10849,AAA,2013J,345357,0,14
10850,AAA,2013J,345357,0,5
10851,AAA,2013J,345357,0,1


In [14]:
print("early_student_vle date minimum:", early_student_vle["date"].min())
print("early_student_vle date maximum:", early_student_vle["date"].max())

print("\nRows in early_student_vle:")
print(early_student_vle.shape[0])

print("\nTotal clicks in early_student_vle:")
print(early_student_vle["sum_click"].sum())

early_student_vle date minimum: 0
early_student_vle date maximum: 30

Rows in early_student_vle:
2291587

Total clicks in early_student_vle:
8431815


In [15]:
daily_vle_activity = (
    early_student_vle
    .groupby(main_keys + ["date"], as_index=False)
    .agg(
        daily_clicks=("sum_click", "sum")
    )
)

print("daily_vle_activity created successfully.")
print("Shape:", daily_vle_activity.shape)

daily_vle_activity.head()

daily_vle_activity created successfully.
Shape: (331688, 5)


,id_student,code_module,code_presentation,date,daily_clicks
0,6516,AAA,2014J,0,71
1,6516,AAA,2014J,1,45
2,6516,AAA,2014J,2,60
3,6516,AAA,2014J,3,17
4,6516,AAA,2014J,5,23


In [16]:
early_vle_features = (
    daily_vle_activity
    .groupby(main_keys, as_index=False)
    .agg(
        total_clicks_30d=("daily_clicks", "sum"),
        active_days_30d=("date", "nunique"),
        avg_clicks_per_active_day_30d=("daily_clicks", "mean"),
        max_clicks_single_day_30d=("daily_clicks", "max"),
        std_clicks_active_day_30d=("daily_clicks", "std"),
        first_activity_day_30d=("date", "min"),
        last_activity_day_30d=("date", "max")
    )
)

early_vle_features["std_clicks_active_day_30d"] = (
    early_vle_features["std_clicks_active_day_30d"].fillna(0)
)

early_vle_features["activity_span_30d"] = (
    early_vle_features["last_activity_day_30d"] 
    - early_vle_features["first_activity_day_30d"] 
    + 1
)

early_vle_features["days_since_last_activity_30d"] = (
    EARLY_WINDOW_END - early_vle_features["last_activity_day_30d"]
)

print("early_vle_features created successfully.")
print("Shape:", early_vle_features.shape)

early_vle_features.head()

early_vle_features created successfully.
Shape: (27971, 12)


,id_student,code_module,code_presentation,total_clicks_30d,active_days_30d,avg_clicks_per_active_day_30d,max_clicks_single_day_30d,std_clicks_active_day_30d,first_activity_day_30d,last_activity_day_30d,activity_span_30d,days_since_last_activity_30d
0,6516,AAA,2014J,578,21,27.523810,142,32.795760,0,30,31,0
1,8462,DDD,2013J,328,17,19.294118,136,33.112997,0,30,31,0
2,8462,DDD,2014J,10,1,10.000000,10,0.000000,10,10,1,20
3,11391,AAA,2013J,326,9,36.222222,127,39.524606,0,30,31,0
4,23629,BBB,2013B,39,5,7.800000,14,4.549725,2,21,20,9


In [17]:
early_vle_total_rows = early_vle_features.shape[0]
early_vle_unique_keys = early_vle_features[main_keys].drop_duplicates().shape[0]

print("early_vle_features total rows:", early_vle_total_rows)
print("early_vle_features unique student-module-presentation keys:", early_vle_unique_keys)
print("Is the main key unique in early_vle_features?", early_vle_total_rows == early_vle_unique_keys)

early_vle_features total rows: 27971
early_vle_features unique student-module-presentation keys: 27971
Is the main key unique in early_vle_features? True


In [18]:
vle_feature_columns = [
    "total_clicks_30d",
    "active_days_30d",
    "avg_clicks_per_active_day_30d",
    "max_clicks_single_day_30d",
    "std_clicks_active_day_30d",
    "activity_span_30d",
    "days_since_last_activity_30d"
]

early_vle_features[vle_feature_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
total_clicks_30d,27971.0,301.448464,354.656335,1.0,77.000000,191.000000,401.000000,6571.000000
active_days_30d,27971.0,11.858282,7.709422,1.0,6.000000,11.000000,17.000000,31.000000
avg_clicks_per_active_day_30d,27971.0,21.986130,15.377306,1.0,11.600000,18.600000,28.615385,260.947368
max_clicks_single_day_30d,27971.0,71.522112,66.468966,1.0,29.000000,56.000000,99.000000,4114.000000
std_clicks_active_day_30d,27971.0,21.202658,18.316825,0.0,9.276921,17.329633,29.091481,934.580742
activity_span_30d,27971.0,23.738265,8.587123,1.0,21.000000,27.000000,30.000000,31.000000
days_since_last_activity_30d,27971.0,4.067069,6.372094,0.0,0.000000,1.000000,5.000000,30.000000


In [19]:
vle_feature_coverage = base_student_dataset[main_keys].merge(
    early_vle_features[main_keys],
    on=main_keys,
    how="left",
    indicator=True
)

print("VLE feature coverage against base_student_dataset:")
print(vle_feature_coverage["_merge"].value_counts())

VLE feature coverage against base_student_dataset:
_merge
both          27971
left_only      4622
right_only        0
Name: count, dtype: int64


In [20]:
early_vle_features.sort_values(
    by=["active_days_30d", "total_clicks_30d"],
    ascending=True
).head(10)

,id_student,code_module,code_presentation,total_clicks_30d,active_days_30d,avg_clicks_per_active_day_30d,max_clicks_single_day_30d,std_clicks_active_day_30d,first_activity_day_30d,last_activity_day_30d,activity_span_30d,days_since_last_activity_30d
180,50610,DDD,2013J,1,1,1.0,1,0.0,8,8,1,22
481,89778,BBB,2014B,1,1,1.0,1,0.0,14,14,1,16
1491,197536,FFF,2014J,1,1,1.0,1,0.0,22,22,1,8
1685,234068,CCC,2014B,1,1,1.0,1,0.0,12,12,1,18
1904,250092,CCC,2014B,1,1,1.0,1,0.0,10,10,1,20
1926,252652,BBB,2014B,1,1,1.0,1,0.0,21,21,1,9
1997,260062,CCC,2014B,1,1,1.0,1,0.0,14,14,1,16
2167,272704,FFF,2014J,1,1,1.0,1,0.0,13,13,1,17
2221,277880,AAA,2014J,1,1,1.0,1,0.0,5,5,1,25
2460,293699,AAA,2013J,1,1,1.0,1,0.0,6,6,1,24
